# RAG Prototype 1 — Ingestion

Reorganized into clear sections: imports, configuration, data acquisition, document extraction, and the ingestion run. Chunking comes next.

## 1. Imports

Everything the notebook needs, in one place, instead of scattered import lines showing up wherever they're first used.

In [1]:
import json
import logging
import re
import time
from dataclasses import dataclass, field
from pathlib import Path

import pymupdf
import requests
from docx import Document
from openpyxl import load_workbook


## 2. Configuration & Logging

One `Config` object instead of loose module-level constants. Makes it obvious what's configurable, and means later stages (chunking, embedding) can just take `CONFIG` as an argument instead of relying on notebook globals.

We also set up a real logger here — writing to both the console and a log file in `logs/` — so errors during ingestion are recorded instead of just scrolling past in cell output.

In [2]:
@dataclass
class Config:
    base_dir: Path = field(default_factory=Path.cwd)
    data_subdir: str = "data"
    output_subdir: str = "outputs"
    log_subdir: str = "logs"

    supported_extensions: tuple = (".pdf", ".docx", ".xlsx")
    download_timeout_s: int = 30
    download_retries: int = 3
    download_retry_backoff_s: float = 2.0

    short_page_word_threshold: int = 20

    @property
    def data_dir(self) -> Path:
        return self.base_dir / self.data_subdir

    @property
    def output_dir(self) -> Path:
        return self.base_dir / self.output_subdir

    @property
    def log_dir(self) -> Path:
        return self.base_dir / self.log_subdir

    def ensure_dirs(self):
        for directory in (self.data_dir, self.output_dir, self.log_dir):
            directory.mkdir(exist_ok=True, parents=True)


CONFIG = Config()
CONFIG.ensure_dirs()

print(f"Base directory: {CONFIG.base_dir}")
print(f"Data directory: {CONFIG.data_dir}")
print(f"Output directory: {CONFIG.output_dir}")
print(f"Log directory: {CONFIG.log_dir}")


Base directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice
Data directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\data
Output directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\outputs
Log directory: c:\Users\Swapn\OneDrive\Documents\GitHub\Surfprice\logs


In [3]:
logger = logging.getLogger("rag_ingestion")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if this cell is re-run

_formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(CONFIG.log_dir / "ingestion.log")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.info("Logger initialized.")


2026-08-26 14:27:54,912 | INFO | Logger initialized.


## 3. Data Inventory

Quick sanity check on what's actually sitting in `data/` before we try to ingest it — useful since files can land there from other sources too (like `rag.xlsx` below), not just from `download_sources`.

In [5]:
def list_data_files(config: Config = CONFIG) -> list[Path]:
    files = sorted(config.data_dir.iterdir())

    logger.info(f"Files in data directory: {len(files)}")
    for file in files:
        size_kb = file.stat().st_size / 1024
        logger.info(f"- {file.name} | {file.suffix or 'no ext'} | {size_kb:.2f} KB")

    return files


data_files = list_data_files()


2026-08-26 14:28:45,246 | INFO | Files in data directory: 4
2026-08-26 14:28:45,248 | INFO | - Attention_Is_All_You_Need.docx | .docx | 330.47 KB
2026-08-26 14:28:45,249 | INFO | - Foundation-LLMs.pdf | .pdf | 2656.22 KB
2026-08-26 14:28:45,251 | INFO | - rag.xlsx | .xlsx | 898.20 KB
2026-08-26 14:28:45,252 | INFO | - Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx | .docx | 245.31 KB


## 4. Document Extraction

`clean_text()` and `compute_stats()` are shared helpers used by every extractor, so every record — regardless of source type — ends up with the same cleaned `text` and the same `char_count` / `word_count` / `line_count` fields. That's what fixes the `KeyError: 'word_count'` from the original notebook: those stats used to only get attached to one early prototype list, not the final merged `documents`.

Each extractor also uses a consistent `location` field (e.g. `page_3`, `paragraph_12`, `Sheet1!row_40`) instead of a per-type key, so downstream code doesn't need to branch on source type just to find out where a chunk came from.

In [6]:
def clean_text(text: str) -> str:
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Collapse runs of spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)
    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def compute_stats(text: str) -> dict:
    return {
        "char_count": len(text),
        "word_count": len(text.split()),
        "line_count": len(text.splitlines()),
    }


def make_record(document: str, source_type: str, location: str, text: str, metadata: dict | None = None) -> dict:
    cleaned = clean_text(text)
    record = {
        "document": document,
        "source_type": source_type,
        "location": location,
        "text": cleaned,
        "metadata": metadata or {},
    }
    record.update(compute_stats(cleaned))
    return record


In [7]:
def extract_pdf(file_path: Path) -> list[dict]:
    documents = []

    with pymupdf.open(file_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="pdf",
                    location=f"page_{page_number}",
                    text=page.get_text(),
                )
            )

    return documents


In [8]:
def extract_docx(file_path: Path) -> list[dict]:
    documents = []

    doc = Document(file_path)

    for index, paragraph in enumerate(doc.paragraphs, start=1):
        text = paragraph.text.strip()

        if text:
            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="docx",
                    location=f"paragraph_{index}",
                    text=text,
                )
            )

    return documents


In [9]:
def extract_xlsx(file_path: Path) -> list[dict]:
    documents = []

    workbook = load_workbook(file_path, read_only=True, data_only=True)

    for sheet in workbook.worksheets:
        for row_number, row in enumerate(sheet.iter_rows(values_only=True), start=1):
            values = [str(value) for value in row if value is not None]

            if not values:
                continue

            text = " | ".join(values)

            documents.append(
                make_record(
                    document=file_path.name,
                    source_type="xlsx",
                    location=f"{sheet.title}!row_{row_number}",
                    text=text,
                    metadata={"sheet": sheet.title, "row": row_number},
                )
            )

    return documents


In [10]:
EXTRACTORS = {
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".xlsx": extract_xlsx,
}


def ingest_file(file_path: Path) -> list[dict]:
    suffix = file_path.suffix.lower()

    extractor = EXTRACTORS.get(suffix)
    if extractor is None:
        raise ValueError(f"Unsupported file format: {suffix}")

    return extractor(file_path)


## 6. Run Ingestion

Each file is wrapped in its own `try/except` so one corrupt or unsupported file logs an error and gets skipped, rather than crashing the whole ingestion run.

In [11]:
def run_ingestion(config: Config = CONFIG) -> list[dict]:
    all_documents = []

    for file_path in sorted(config.data_dir.iterdir()):
        if file_path.suffix.lower() not in config.supported_extensions:
            continue

        try:
            extracted = ingest_file(file_path)
            all_documents.extend(extracted)
            logger.info(f"{file_path.name}: {len(extracted)} records")

        except Exception:
            logger.exception(f"Failed to ingest {file_path.name}, skipping.")

    logger.info(f"Total records: {len(all_documents)}")
    return all_documents


documents = run_ingestion()


2026-08-26 14:30:21,517 | INFO | Attention_Is_All_You_Need.docx: 298 records
2026-08-26 14:30:22,651 | INFO | Foundation-LLMs.pdf: 277 records
2026-08-26 14:30:23,533 | INFO | rag.xlsx: 4720 records
2026-08-26 14:30:23,761 | INFO | Retrieval-Augmented_Generation_for_Knowledge-Intensive_NLP_Tasks.docx: 211 records
2026-08-26 14:30:23,763 | INFO | Total records: 5506


### Sanity checks

Same checks as before, fixed to use the fields every record now actually has (`location` instead of `page`, and `word_count`/`text` present on every record).

In [12]:
empty_docs = [doc for doc in documents if not doc["text"]]

print(f"Total records: {len(documents)}")
print(f"Empty records: {len(empty_docs)}")
print(f"Non-empty records: {len(documents) - len(empty_docs)}")


Total records: 5506
Empty records: 0
Non-empty records: 5506


In [13]:
short_docs = [
    doc for doc in documents
    if doc["word_count"] < CONFIG.short_page_word_threshold
]

print(f"Suspiciously short records (< {CONFIG.short_page_word_threshold} words): {len(short_docs)}")

for doc in short_docs[:10]:
    print(doc["document"], "|", doc["location"], "| words:", doc["word_count"])


Suspiciously short records (< 20 words): 516
Attention_Is_All_You_Need.docx | paragraph_3 | words: 5
Attention_Is_All_You_Need.docx | paragraph_10 | words: 5
Attention_Is_All_You_Need.docx | paragraph_11 | words: 5
Attention_Is_All_You_Need.docx | paragraph_12 | words: 5
Attention_Is_All_You_Need.docx | paragraph_13 | words: 5
Attention_Is_All_You_Need.docx | paragraph_17 | words: 5
Attention_Is_All_You_Need.docx | paragraph_18 | words: 8
Attention_Is_All_You_Need.docx | paragraph_19 | words: 2
Attention_Is_All_You_Need.docx | paragraph_20 | words: 2
Attention_Is_All_You_Need.docx | paragraph_21 | words: 1


Next up: chunking these `documents` records before they go into the vector store.

In [14]:
for source_type in ["pdf", "docx", "xlsx"]:

    matches = [
        doc for doc in documents
        if doc["source_type"] == source_type
    ]

    if matches:
        print(f"\n{'=' * 70}")
        print(source_type.upper())
        print(f"{'=' * 70}")
        print(matches[0])


PDF
{'document': 'Foundation-LLMs.pdf', 'source_type': 'pdf', 'location': 'page_1', 'text': 'arXiv:2501.09223v2 [cs.CL] 15 Jun 2025\nFoundations of\nLarge Language Models\nTong Xiao and Jingbo Zhu\nJune 17, 2025\nNLP Lab, Northeastern University & NiuTrans Research\nThis book is a selection of chapters from an introductory NLP resource\navailable at https://github.com/NiuTrans/NLPBook', 'metadata': {}, 'char_count': 287, 'word_count': 40, 'line_count': 8}

DOCX
{'document': 'Attention_Is_All_You_Need.docx', 'source_type': 'docx', 'location': 'paragraph_1', 'text': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', 'metadata': {}, 'char_count': 173, 'word_count': 26, 'line_count': 1}

XLSX
{'document': 'rag.xlsx', 'source_type': 'xlsx', 'location': 'Sheet1!row_1', 'text': 'question | answer | relevant_passage_ids | id', 'metadata': {'sheet': 'Sheet1', 'row': 1},

In [15]:
def inspect_xlsx_schema(file_path):

    workbook = load_workbook(
        file_path,
        read_only=True,
        data_only=True
    )

    schema = {}

    for sheet in workbook.worksheets:

        rows = list(sheet.iter_rows(values_only=True))

        if not rows:
            continue

        headers = [
            str(value).strip()
            if value is not None
            else f"column_{i}"
            for i, value in enumerate(rows[0])
        ]

        columns = {
            header: []
            for header in headers
        }

        for row in rows[1:]:
            for header, value in zip(headers, row):
                columns[header].append(value)

        schema[sheet.title] = columns

    workbook.close()

    return schema

In [ ]:
xlsx_files = list(config.data_dir.glob("*.xlsx"))

for file_path in xlsx_files:

    schema = inspect_xlsx_schema(file_path)

    print(f"\n{'=' * 70}")
    print(file_path.name)
    print(f"{'=' * 70}")

    for sheet_name, columns in schema.items():

        print(f"\nSheet: {sheet_name}")

        for column_name, values in columns.items():
            print(
                f"{column_name}: "
                f"{len(values)} values"
            )

In [ ]:
SUSPICIOUS_NAME_TERMS = {
    "id",
    "row_id",
    "index",
    "rank",
    "ranking",
    "serial",
    "sr_no",
    "s_no"
}


def name_signal(column_name):
    name = column_name.lower().strip()

    if name in SUSPICIOUS_NAME_TERMS:
        return 1.0

    return 0.0

In [ ]:
def calculate_column_stats(values):

    total = len(values)

    non_empty = [
        value for value in values
        if value is not None and str(value).strip() != ""
    ]

    if not non_empty:
        return {
            "total": total,
            "non_empty": 0,
            "unique": 0,
            "unique_ratio": 0,
            "repetition_ratio": 0,
            "missing_ratio": 1.0
        }

    unique_values = set(
        str(value).strip()
        for value in non_empty
    )

    non_empty_count = len(non_empty)
    unique_count = len(unique_values)

    return {
        "total": total,
        "non_empty": non_empty_count,
        "unique": unique_count,
        "unique_ratio": unique_count / non_empty_count,
        "repetition_ratio": 1 - (
            unique_count / non_empty_count
        ),
        "missing_ratio": (
            (total - non_empty_count) / total
            if total else 0
        )
    }

In [ ]:
def classify_column(column_name, values):

    stats = calculate_column_stats(values)

    name_score = name_signal(column_name)

    unique_score = (
        1.0
        if stats["unique_ratio"] >= 0.95
        else 0.0
    )

    repetition_score = (
        1.0
        if stats["repetition_ratio"] >= 0.90
        else 0.0
    )

    missing_score = (
        1.0
        if stats["missing_ratio"] >= 0.80
        else 0.0
    )

    metadata_score = (
        name_score * 0.50
        + unique_score * 0.30
        + missing_score * 0.20
    )

    redundancy_score = repetition_score

    if metadata_score >= 0.5:
        classification = "potential_metadata"

    elif redundancy_score >= 0.9:
        classification = "potential_redundancy"

    else:
        classification = "likely_semantic"

    return {
        **stats,
        "name_score": name_score,
        "metadata_score": round(metadata_score, 3),
        "redundancy_score": round(redundancy_score, 3),
        "classification": classification
    }

In [ ]:
for file_path in xlsx_files:

    schema = inspect_xlsx_schema(file_path)

    print(f"\n{'=' * 70}")
    print(file_path.name)
    print(f"{'=' * 70}")

    for sheet_name, columns in schema.items():

        print(f"\nSheet: {sheet_name}")

        for column_name, values in columns.items():

            result = classify_column(
                column_name,
                values
            )

            print(
                f"{column_name:<25} "
                f"→ {result['classification']:<20} "
                f"metadata={result['metadata_score']:.2f} "
                f"redundancy={result['redundancy_score']:.2f}"
            )

In [ ]:
column_analysis = []

for file_path in xlsx_files:

    schema = inspect_xlsx_schema(file_path)

    for sheet_name, columns in schema.items():

        for column_name, values in columns.items():

            result = classify_column(
                column_name,
                values
            )

            column_analysis.append({
                "file": file_path.name,
                "sheet": sheet_name,
                "column": column_name,
                "classification": result["classification"],
                "unique_ratio": round(
                    result["unique_ratio"], 3
                ),
                "repetition_ratio": round(
                    result["repetition_ratio"], 3
                ),
                "missing_ratio": round(
                    result["missing_ratio"], 3
                ),
                "metadata_score": result["metadata_score"],
                "redundancy_score": result["redundancy_score"]
            })

In [ ]:
import pandas as pd

analysis_df = pd.DataFrame(column_analysis)

analysis_df

In [ ]:
analysis_df[
    analysis_df["classification"] != "likely_semantic"
]

In [ ]:
for _, row in analysis_df[
    analysis_df["classification"] != "likely_semantic"
].iterrows():

    file_path = DATA_DIR / row["file"]

    schema = inspect_xlsx_schema(file_path)

    values = schema[row["sheet"]][row["column"]]

    print("\n" + "=" * 70)
    print(f"Column: {row['column']}")
    print(f"Classification: {row['classification']}")
    print("Sample values:")

    for value in values[:10]:
        print(f"  {value}")

In [ ]:
def create_normalized_record(
    document,
    source_type,
    location,
    text,
    metadata=None
):
    return {
        "document": document,
        "source_type": source_type,
        "location": location,
        "text": text,
        "metadata": metadata or {}
    }

In [ ]:
normalized_documents = []

for record in documents:

    normalized_record = create_normalized_record(
        document=record["document"],
        source_type=record["source_type"],
        location=record["location"],
        text=record["text"],
        metadata=record.get("metadata", {})
    )

    normalized_documents.append(normalized_record)

print(
    f"Normalized records: "
    f"{len(normalized_documents)}"
)

In [ ]:
required_fields = {
    "document",
    "source_type",
    "location",
    "text",
    "metadata"
}

invalid_records = []

for index, record in enumerate(normalized_documents):

    if set(record.keys()) != required_fields:
        invalid_records.append(index)

print(f"Total records: {len(normalized_documents)}")
print(f"Invalid records: {len(invalid_records)}")

In [ ]:
for source_type in ["pdf", "docx", "xlsx"]:

    records = [
        record
        for record in normalized_documents
        if record["source_type"] == source_type
    ]

    if records:
        print(f"\n{source_type.upper()}")
        print("-" * 50)
        print(records[0])

In [ ]:
for record in normalized_documents:
    record["raw_text"] = record["text"]

In [ ]:
import re


def normalize_text(text: str) -> str:

    if not text:
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Replace tabs with spaces
    text = text.replace("\t", " ")

    # Remove excessive spaces
    text = re.sub(r"[ ]{2,}", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

In [ ]:
for record in normalized_documents:

    record["text"] = normalize_text(
        record["text"]
    )

In [ ]:
for record in normalized_documents[:5]:

    print("=" * 80)
    print(
        record["document"],
        "|",
        record["location"]
    )

    print("\nRAW:")
    print(repr(record["raw_text"][:500]))

    print("\nCLEANED:")
    print(repr(record["text"][:500]))

In [ ]:
total_raw_chars = sum(
    len(record["raw_text"])
    for record in normalized_documents
)

total_clean_chars = sum(
    len(record["text"])
    for record in normalized_documents
)

removed = total_raw_chars - total_clean_chars

print(f"Raw characters:     {total_raw_chars:,}")
print(f"Cleaned characters: {total_clean_chars:,}")
print(f"Characters removed: {removed:,}")

In [ ]:
empty_after_cleaning = [
    record
    for record in normalized_documents
    if not record["text"].strip()
]

print(
    "Empty records after cleaning:",
    len(empty_after_cleaning)
)

In [ ]:
short_records = [
    record
    for record in normalized_documents
    if len(record["text"].split()) < 5
]

print(
    "Records with fewer than 5 words:",
    len(short_records)
)

for record in short_records[:10]:
    print(
        record["source_type"],
        "|",
        record["document"],
        "|",
        record["location"],
        "|",
        repr(record["text"])
    )

In [ ]:
from collections import Counter


def get_pdf_lines(records):

    lines = []

    for record in records:
        if record["source_type"] != "pdf":
            continue

        page_lines = [
            line.strip()
            for line in record["raw_text"].splitlines()
            if line.strip()
        ]

        lines.append(page_lines)

    return lines

In [ ]:
pdf_records = [
    record
    for record in normalized_documents
    if record["source_type"] == "pdf"
]

line_counter = Counter()

for record in pdf_records:

    lines = {
        line.strip()
        for line in record["raw_text"].splitlines()
        if line.strip()
    }

    line_counter.update(lines)

In [ ]:
for line, count in line_counter.most_common(20):

    if count > 1:
        print(f"{count:>3}x | {line}")

In [ ]:
docx_records = [
    record
    for record in normalized_documents
    if record["source_type"] == "docx"
]

for record in docx_records[:20]:

    print(
        record["location"],
        "|",
        repr(record["text"][:150])
    )

In [ ]:
def extract_docx(file_path):

    records = []

    doc = Document(file_path)

    for index, paragraph in enumerate(
        doc.paragraphs,
        start=1
    ):

        text = paragraph.text.strip()

        if not text:
            continue

        records.append({
            "document": file_path.name,
            "source_type": "docx",
            "location": f"paragraph_{index}",
            "text": text,
            "metadata": {
                "style": paragraph.style.name
            }
        })

    return records

In [ ]:
xlsx_column_flags = []

for file_path in xlsx_files:

    schema = inspect_xlsx_schema(file_path)

    for sheet_name, columns in schema.items():

        for column_name, values in columns.items():

            result = classify_column(
                column_name,
                values
            )

            xlsx_column_flags.append({
                "file": file_path.name,
                "sheet": sheet_name,
                "column": column_name,
                "classification": result["classification"],
                "metadata_score": result["metadata_score"],
                "redundancy_score": result["redundancy_score"]
            })

In [ ]:
xlsx_flags_df = pd.DataFrame(
    xlsx_column_flags
)

xlsx_flags_df